In [5]:
# ==========================================
# STEP 1: LIBRARY IMPORT & FILE UPLOAD
# ==========================================

import pandas as pd
import io
from google.colab import files

print("Please upload the project dataset (.xlsx file).")
print("Click 'Choose Files' below:")

# Trigger the file upload widget
# This allows the user to interactively select the file from their local machine
uploaded = files.upload()

# Get the name of the uploaded file dynamically
# This prevents errors if the filename changes in future iterations
file_path = next(iter(uploaded))
print(f"\n[INFO] File '{file_path}' uploaded successfully.")

# ==========================================
# STEP 2: DATA INGESTION (READING SHEETS)
# ==========================================

# Inspect the workbook to understand the data organization
xls = pd.ExcelFile(file_path)

print("\n[INFO] Available Sheets in Workbook:")
print(xls.sheet_names)

# Loading specific sheets into separate Pandas DataFrames based on the project structure
# 'Procurement Data' acts as the Fact Table (Transactions)
# 'Agents', 'Suppliers', and 'Products' act as Dimension Tables (Master Data)

try:
    df_procurement = pd.read_excel(file_path, sheet_name='Procurement Data 23-24')
    df_agents = pd.read_excel(file_path, sheet_name='Agents')
    df_suppliers = pd.read_excel(file_path, sheet_name='Suppliers')
    df_products = pd.read_excel(file_path, sheet_name='Products')
    print("\n[SUCCESS] All data sheets loaded successfully.")

except ValueError as e:
    print(f"\n[ERROR] Could not load sheets. Please check sheet names. Error details: {e}")

# ==========================================
# STEP 3: INITIAL DATA INSPECTION
# ==========================================

# Display the first 5 rows of each DataFrame to verify correct header alignment

print("\n====== PROCUREMENT DATA (TRANSACTIONS) HEAD ======")
print(df_procurement.head())

print("\n====== AGENTS TABLE HEAD ======")
print(df_agents.head())

print("\n====== SUPPLIERS TABLE HEAD ======")
print(df_suppliers.head())

print("\n====== PRODUCTS TABLE HEAD ======")
print(df_products.head())

Please upload the project dataset (.xlsx file).
Click 'Choose Files' below:


Saving Velocipede Cycles 23-24 Dataset.xlsx to Velocipede Cycles 23-24 Dataset.xlsx

[INFO] File 'Velocipede Cycles 23-24 Dataset.xlsx' uploaded successfully.

[INFO] Available Sheets in Workbook:
['Data Dictionary', 'Procurement Data 23-24', 'Agents', 'Suppliers', 'Products']

[SUCCESS] All data sheets loaded successfully.

====== PROCUREMENT DATA (TRANSACTIONS) HEAD ======
  PO Number PO Month  PO DoM Buyer Agent Item_ID Supplier  \
0  PO006670   Oct-24       1      BA-221  PRD016   SUP391   
1  PO000438   Jun-23       1      BA-221  PRD012   SUP128   
2  PO002662   Oct-24       1      BA-102  PRD026   SUP072   
3  PO005520   Dec-24       1      BA-481  PRD038   SUP189   
4  PO004963   Jul-24       1      BA-481  PRD009   SUP056   

   Average Buying Price  PO Amount  PO Quantity  Suppliers OTP  
0                164.35    3287.00         20.0              1  
1               1060.00   10600.00         10.0              1  
2                 60.80     608.00         10.0             

In [6]:
# ==========================================
# STEP 4: DATA QUALITY ASSESSMENT
# ==========================================

print("performing data quality checks...")

# Check for Missing Values in the main transaction table
missing_values = df_procurement.isnull().sum()
print("\n[INFO] Missing Values per Column in Procurement Data:")
print(missing_values[missing_values > 0])

# Check for Duplicate Rows
duplicates = df_procurement.duplicated().sum()
print(f"\n[INFO] Duplicate Rows found in Transaction Data: {duplicates}")

# Note: If significant missing values are found, imputation strategies would be applied here.
# For this dataset, we proceed assuming minor data gaps are acceptable or handled in visualization.

# ==========================================
# STEP 5: DATA INTEGRATION (MERGING)
# ==========================================

# Objective: Enrich the transaction data (Fact Table) with attributes from Dimension Tables.
# We use 'Left Joins' to ensure all transaction records are retained.

# 1. Join with Products Table (Get Product Name, Category, UoM)
df_master = pd.merge(df_procurement, df_products,
                     left_on='Item_ID', right_on='Product_ID',
                     how='left')

# 2. Join with Suppliers Table (Get Supplier Name, Risk Notes)
df_master = pd.merge(df_master, df_suppliers,
                     left_on='Supplier', right_on='Supplier_ID',
                     how='left')

# 3. Join with Agents Table (Get Buyer Agent Name)
df_master = pd.merge(df_master, df_agents,
                     left_on='Buyer Agent', right_on='Agent Code',
                     how='left')

# ==========================================
# STEP 6: DATA CLEANING & TRANSFORMATION
# ==========================================

# 1. Convert 'PO Month' string (e.g., 'Oct-24') to Datetime format
# This is crucial for Time-Series analysis in Tableau
df_master['Order_Date'] = pd.to_datetime(df_master['PO Month'], format='%b-%y')

# 2. Drop Redundant ID Columns
# Since we have the joined data, we can remove duplicate ID columns to keep the dataset clean
columns_to_drop = ['Product_ID', 'Supplier_ID', 'Agent Code', 'PO Month']
df_master.drop(columns=columns_to_drop, axis=1, inplace=True, errors='ignore')

# 3. Reorder Columns for better readability (Optional but good for review)
cols = ['Order_Date', 'PO Number', 'Supplier Name', 'Category', 'Sub-Category',
        'Product Name', 'Buyer Agent Name', 'PO Quantity', 'Average Buying Price', 'PO Amount', 'Suppliers OTP']

# Reindex with available columns only (prevents errors if a column is missing)
existing_cols = [c for c in cols if c in df_master.columns]
df_master = df_master[existing_cols + [c for c in df_master.columns if c not in existing_cols]]

# Check final Data Structure
print("\n[SUCCESS] Master Table Created Successfully.")
print(f"Total Records: {df_master.shape[0]}")
print(f"Total Columns: {df_master.shape[1]}")

print("\n====== MASTER DATA PREVIEW (FIRST 5 ROWS) ======")
print(df_master.head())

performing data quality checks...

[INFO] Missing Values per Column in Procurement Data:
Series([], dtype: int64)

[INFO] Duplicate Rows found in Transaction Data: 3

[SUCCESS] Master Table Created Successfully.
Total Records: 4995
Total Columns: 18

====== MASTER DATA PREVIEW (FIRST 5 ROWS) ======
  Order_Date PO Number              Supplier Name               Category  \
0 2024-10-01  PO006670             Redgum Systems             Wheel Sets   
1 2023-06-01  PO000438          Tasmanian Traders             Wheel Sets   
2 2024-10-01  PO002662   Coral Coast Technologies  Handlebar Accessories   
3 2024-12-01  PO005520  Harbour City Distributors             Drivetrain   
4 2024-07-01  PO004963    Great Southern Supplies           Brake System   

      Sub-Category          Product Name Buyer Agent Name  PO Quantity  \
0  Complete Wheels  Complete Road Wheels   Amina El-Sayed         20.0   
1    Carbon Wheels      Carbon Wheel Set   Amina El-Sayed         10.0   
2        Handlebar   

In [7]:
# ==========================================
# STEP 7: EXPORT DATA FOR VISUALIZATION
# ==========================================

output_filename = 'Velocipede_Cleaned_Master_Data.csv'

# Save to CSV without the pandas index number
df_master.to_csv(output_filename, index=False)

print(f"\n[INFO] Processed data saved as '{output_filename}'")
print("Downloading file for Tableau...")

# Trigger download in Google Colab
files.download(output_filename)


[INFO] Processed data saved as 'Velocipede_Cleaned_Master_Data.csv'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>